# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedr456/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring** (provisional).

Why this one: it is the lane where I already have evidence in hand. In Week 1 I ran the
reference pipeline (`notebooks/01_first_look_and_discovery.ipynb`) and saw a learned model
roughly **3x** a hand-written refresh rule on Precision@50, and my own discovery cell found
that CTR differs by `content_type` even at the same position tier — exactly the kind of
signal a refresh queue can turn into reason codes. The lane also has the deepest data
support (starter playground now, warehouse daily facts later for stronger time-window
labels), so every later assignment (contract → baseline → model → validation → action
queue) has a clear home.

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mohamedr456/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages | {df['client_id'].nunique()} clients | "
      f"declining base rate: {(df['trend_direction'].str.lower()=='down').mean():.3f}")

30,000 pages | 32 clients | declining base rate: 0.542


## 2. The question: decision, action, cost of a wrong call

**Research question:** *Which pages should a content editor review first for refresh,
expansion, protection, pruning, or monitoring?*

- **Decision improved:** how an editor spends their limited review hours each week — which
  pages to open first, out of tens of thousands.
- **Who acts:** a FlyRank content strategist working a per-client review queue. They open
  the page, read the reason code, and choose an action (refresh / expand / protect / prune
  / keep monitoring).
- **Cost of a wrong call:** a *false positive* wastes an editor-hour on a healthy page
  (queue slots are scarce — a top-50 slot given to a fine page is a declining page not
  looked at). A *false negative* lets a high-traffic page decay silently — the costlier
  error, since declining pages hold about half of all impressions in this panel.
- **Why ML at all:** the hand rule below (stale AND visible) surfaces only **17 of 30,000
  pages** — it starves the queue and its top-50 is mostly unscored ties. The pattern
  (staleness × visibility × position × engagement × content type) is real but too tangled
  to hand-write; that is exactly where a learned, *readable* scorer earns its keep.

In [2]:
# Back the framing with numbers: the hand rule starves the queue.
y = df["trend_direction"].str.lower().eq("down").astype(int).values

stale   = (df["days_since_last_update"] >= 180)
visible = (df["impressions_90d"] >= 500)
rule_hits = int((stale & visible).sum())

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

rule_score = (stale & visible).astype(int) * df["impressions_90d"]
print(f"Pages passing the hand rule (stale>=180d AND impressions>=500): {rule_hits} of {len(df):,}")
print(f"Hand-rule Precision@50: {precision_at_k(rule_score, y, 50):.3f}"
      f"   vs declining base rate: {y.mean():.3f}")
print("-> a top-50 queue built from 17 scored pages + ties is not a usable review queue.")

Pages passing the hand rule (stale>=180d AND impressions>=500): 17 of 30,000
Hand-rule Precision@50: 0.620   vs declining base rate: 0.542
-> a top-50 queue built from 17 scored pages + ties is not a usable review queue.


## 3. Quick look at the data (2-3 real numbers)

Three numbers, computed live below, that make this lane worth the next 7 weeks:

1. **54.2% of the 30,000 pages are declining** — refresh triage is not a rare-event hunt,
   it is the daily reality of this panel.
2. **Declining pages hold ~51% of all impressions** — the traffic at stake is real, not a
   long tail of dead pages.
3. **The reference pipeline already showed the gap is closable:** hand rule 0.240 vs
   random forest 0.740 Precision@50 (client-holdout split) in
   `notebooks/01_first_look_and_discovery.ipynb` — evidence a learned queue is worth
   building properly, with honest validation.

In [3]:
# The three numbers, computed from the shipped CSV (nothing hardcoded).
declining_share = y.mean()
imp_share_declining = df.loc[y == 1, "impressions_90d"].sum() / df["impressions_90d"].sum()
print(f"1. Declining share of pages:          {declining_share:.3f}")
print(f"2. Impressions held by declining:     {imp_share_declining:.3f}")

# outputs/model_results.json is a regenerated artifact (gitignored) — rebuild it
# by running the reference pipeline if it isn't there (~1 minute).
import json, subprocess
if not os.path.exists("outputs/model_results.json"):
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True,
                   stdout=subprocess.DEVNULL)
res = json.load(open("outputs/model_results.json"))
print(f"3. Pipeline gap (client-holdout):     rule P@50 "
      f"{res['baseline']['baseline_precision_at_50']:.3f}  vs  "
      f"random forest {res['models']['random_forest']['precision_at_50']:.3f}")

1. Declining share of pages:          0.542
2. Impressions held by declining:     0.513


3. Pipeline gap (client-holdout):     rule P@50 0.240  vs  random forest 0.740


## 4. Careful words: what I can and can't claim

**I can say (and will phrase this way):**
- *Observed:* metric differences measured in this pseudonymized 90-day panel
  (e.g. "declining pages hold ~51% of impressions in this sample").
- *Directional:* patterns that repeat across clients and survive a client-holdout split
  (e.g. "CTR differs by content type at the same position tier — directionally").
- *Decision-support:* "this queue ordering put ~3x more truly-declining pages in the top
  50 than the hand rule, on held-out clients" — a claim about *review priority*, not about
  outcomes of acting.

**I will never claim:**
- **Causation** — no "refreshing causes recovery"; I have no experiment, only observation.
- **"Predicting Google"** — the data observes one panel's search performance; it does not
  reverse-engineer any ranking algorithm.
- **Anything client-identifying** — IDs are pseudonyms and stay out of features and prose.

**Standing data-honesty guards (from `docs/data-dictionary.md` gotchas):**
rate columns are ×100 percentages; `avg_position = 0` means *no data*, not rank 0; and the
label trap — `trend_direction` / `trend_pct` define the label, so they are **never features**.

In [4]:
# Guard rails, checked in code so Run All fails if I ever break them.
CANDIDATE_FEATURES = ["content_age_days", "days_since_last_update", "impressions_90d",
                      "avg_position", "ctr", "word_count", "engagement_rate"]
LABEL_SOURCES = {"trend_direction", "trend_pct"}
ID_COLUMNS    = {"content_id", "client_id"}

assert not (set(CANDIDATE_FEATURES) & LABEL_SOURCES), "label trap: trend_* leaked into features"
assert not (set(CANDIDATE_FEATURES) & ID_COLUMNS),    "pseudonym IDs must not be features"
assert set(CANDIDATE_FEATURES) <= set(df.columns),     "feature named that doesn't exist"

n_no_position = int((df["avg_position"] == 0).sum())
print(f"Guards pass. Noted: {n_no_position:,} rows have avg_position=0 (= NO DATA, to be flagged, not used as rank 0).")

Guards pass. Noted: 1,205 rows have avg_position=0 (= NO DATA, to be flagged, not used as rank 0).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.